<a href="https://colab.research.google.com/github/Tuyentruong123/skills-copilot-codespaces-vscode/blob/main/Auto_coding_CSAT_Freight.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import torch
import pickle
import os
from google.colab import auth, drive
import gspread
from google.auth import default
from transformers import AutoTokenizer, AutoModel
from sklearn.multioutput import MultiOutputClassifier
from sklearn.linear_model import LogisticRegression

# ==========================================
# 1. KẾT NỐI HỆ THỐNG VÀ ĐỌC FILE GOOGLE SHEET
# ==========================================
print("🔄 Bước 1: Đang kết nối tài khoản, Drive và cấu trúc file Google Sheet...")
drive.mount('/content/drive', force_remount=True)
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

spreadsheet_url = "https://docs.google.com/spreadsheets/d/14dJ7xADbIM4U23gx_syZX2KurtTex7wM0ojvSp752dw/edit"
sh = gc.open_by_url(spreadsheet_url)
sheet_coding = sh.worksheet("Q1b_Q4. OE_coding")

if sheet_coding.col_count < 55:
    sheet_coding.add_cols(55 - sheet_coding.col_count)

all_rows = sheet_coding.get_all_values()
headers = [str(h).strip().lower() for h in all_rows[0]]
df = pd.DataFrame(all_rows[1:], columns=headers)

# ==========================================
# 2. KHỞI TẠO BỘ NÃO NGÔN NGỮ PHOBERT
# ==========================================
print("\n🔥 Bước 2: Đang kích hoạt bộ xử lý ngữ cảnh PhoBERT...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base")
phobert = AutoModel.from_pretrained("vinai/phobert-base").to(device)

def get_bert_embeddings(text_list, batch_size=32):
    all_embeddings = []
    phobert.eval()
    for i in range(0, len(text_list), batch_size):
        batch_texts = text_list[i:i+batch_size]
        encoded_input = tokenizer(batch_texts, padding=True, truncation=True, max_length=128, return_tensors='pt').to(device)
        with torch.no_grad():
            model_output = phobert(**encoded_input)
            embeddings = model_output.last_hidden_state[:, 0, :].cpu().numpy()
            all_embeddings.append(embeddings)
    return np.vstack(all_embeddings)

# Hàm làm sạch mã code gán thực tế trên dòng (Chỉ giữ lại mã >= 100 hoặc mã đặc biệt 999)
def clean_exact_code(val):
    if pd.isna(val) or val is None: return ""
    val_str = str(val).strip()
    if val_str.endswith('.0'): val_str = val_str[:-2]
    if val_str.lower() in ['', 'nan', 'none', 'null'] or not val_str.isdigit():
        return ""

    # ÉP BUỘC CHỈ LẤY CODE >= 100 (Loại bỏ các mã 1, 2, 3...)
    if int(val_str) < 100:
        return ""

    return val_str

# ==========================================
# 3. TIẾN HÀNH TRÍCH XUẤT MÃ VÀ HUẤN LUYỆN (TRAIN)
# ==========================================
print("\n🧠 Bước 3: AI tự động quét dữ liệu đã gán thực tế trên dòng để học...")

def train_pipeline_for_question(text_col, code_cols):
    unique_codes = set()
    for col in code_cols:
        if col in df.columns:
            for val in df[col].unique():
                cleaned = clean_exact_code(val)
                if cleaned != "": unique_codes.add(cleaned)
    all_codes = sorted(list(unique_codes), key=int)

    df_train = df[df[text_col].astype(str).str.strip() != ""].reset_index(drop=True)

    if len(all_codes) == 0 or len(df_train) == 0:
        return None, []

    matrix = np.zeros((len(df_train), len(all_codes)))
    for idx, row in df_train.iterrows():
        row_codes = []
        for col in code_cols:
            if col in df_train.columns:
                cleaned = clean_exact_code(row[col])
                if cleaned != "": row_codes.append(cleaned)
        for c_idx, code in enumerate(all_codes):
            if code in row_codes:
                matrix[idx, c_idx] = 1

    X_emb = get_bert_embeddings(df_train[text_col].astype(str).tolist())

    clf = MultiOutputClassifier(LogisticRegression(max_iter=1000, class_weight='balanced'))
    clf.fit(X_emb, matrix)
    return clf, all_codes

# --- Train cho Q1b ---
code_cols_q1b = ['q1b_1', 'q1b_2', 'q1b_3', 'q1b_4', 'q1b_5', 'q1b_6', 'q1b_7', 'q1b_8', 'q1b_9', 'q1b_10']
print("⏳ Đang phân tích dữ liệu gán thực tế của Q1b...")
clf_q1b, all_codes_q1b = train_pipeline_for_question('q_1b_raw', code_cols_q1b)
print(f"📊 [Q1b] Bộ dải mã thực tế ghi nhận trên các dòng: {all_codes_q1b}")

# --- Train cho Q4 ---
code_cols_q4 = ['q4_1', 'q4_2', 'q4_3', 'q4_4', 'q4_5', 'q4_6', 'q4_7', 'q4_8', 'q4_9', 'q4_10']
print("⏳ Đang phân tích dữ liệu gán thực tế của Q4...")
clf_q4, all_codes_q4 = train_pipeline_for_question('q_4_raw', code_cols_q4)
print(f"📊 [Q4] Bộ dải mã thực tế ghi nhận trên các dòng: {all_codes_q4}")

os.makedirs('/content/drive/MyDrive/GHN_AI_Model', exist_ok=True)
with open('/content/drive/MyDrive/GHN_AI_Model/ghn_oe_ai_model.pkl', 'wb') as f:
    pickle.dump({'clf_q1b': clf_q1b, 'all_codes_q1b': all_codes_q1b, 'clf_q4': clf_q4, 'all_codes_q4': all_codes_q4}, f)

# ==========================================
# 4. TIẾN HÀNH DỰ ĐOÁN ĐA NHÃN VÀ XUẤT 10 CỘT GỢI Ý
# ==========================================
print("\n🚀 Bước 4: AI phân tích diện rộng và đổ dữ liệu gợi ý xuống dải cột AG:AZ...")

def run_prediction_and_write(text_col, question_type, classifier_model, target_labels):
    if classifier_model is None or len(target_labels) == 0:
        print(f"⚠️ Không có mô hình hoặc nhãn hợp lệ cho {question_type}. Bỏ qua bước xuất.")
        return

    if question_type == 'q1b':
        col_letter_start, col_letter_end = "AG", "AP"
        start_col_idx = 33
        print("✍️ Đang ghi chuẩn hóa tiêu đề Q1b (Cột AG:AP)...")
        for i in range(10): sheet_coding.update_cell(1, start_col_idx + i, f"ai gợi ý q1b_{i+1}")
    else:
        col_letter_start, col_letter_end = "AQ", "AZ"
        start_col_idx = 43
        print("✍️ Đang ghi chuẩn hóa tiêu đề Q4 (Cột AQ:AZ)...")
        for i in range(10): sheet_coding.update_cell(1, start_col_idx + i, f"ai gợi ý q4_{i+1}")

    has_text = df[text_col].astype(str).str.strip() != ""
    rows_to_predict = df[has_text]

    if len(rows_to_predict) == 0:
        print(f"✨ Cột dữ liệu {text_col} trống.")
        return

    new_texts = rows_to_predict[text_col].astype(str).tolist()
    new_embeddings = get_bert_embeddings(new_texts)

    pred_proba = classifier_model.predict_proba(new_embeddings)
    total_rows_in_sheet = len(df) + 1
    update_matrix = [[""] * 10 for _ in range(total_rows_in_sheet - 1)]

    for original_idx in range(len(rows_to_predict)):
        predicted_codes = []
        for i, col_proba in enumerate(pred_proba):
            if col_proba[original_idx][1] >= 0.55:
                predicted_codes.append(target_labels[i])

        if len(predicted_codes) == 0:
            predicted_codes = ["Không rõ bối cảnh"]

        row_slots = [""] * 10
        for slot_idx in range(min(10, len(predicted_codes))):
            row_slots[slot_idx] = predicted_codes[slot_idx]

        real_df_index = rows_to_predict.index[original_idx]
        update_matrix[int(real_df_index)] = row_slots

    print(f"⚡ Đang dán khối dữ liệu xuống dải ô {col_letter_start}2:{col_letter_end}{total_rows_in_sheet}...")
    cell_range = f"{col_letter_start}2:{col_letter_end}{total_rows_in_sheet}"
    sheet_coding.update(values=update_matrix, range_name=cell_range)

# Thực thi tạo gợi ý
run_prediction_and_write('q_1b_raw', 'q1b', clf_q1b, all_codes_q1b)
run_prediction_and_write('q_4_raw', 'q4', clf_q4, all_codes_q4)

print("\n🎉 THÀNH CÔNG RỰC RỠ! Toàn bộ dải mã gợi ý Q4 đã được lọc sạch, chỉ hiển thị đầu mã từ 101 trở lên.")

🔄 Bước 1: Đang kết nối tài khoản, Drive và cấu trúc file Google Sheet...
Mounted at /content/drive

🔥 Bước 2: Đang kích hoạt bộ xử lý ngữ cảnh PhoBERT...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.decoder.bias      | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🧠 Bước 3: AI tự động quét dữ liệu đã gán thực tế trên dòng để học...
⏳ Đang phân tích dữ liệu gán thực tế của Q1b...
📊 [Q1b] Bộ dải mã thực tế ghi nhận trên các dòng: ['101', '102', '103', '104', '105', '106', '107', '108', '109', '110', '111', '113', '114', '115', '116', '117', '118', '119', '121', '122', '124', '125', '126', '127', '128', '129', '130', '131', '132', '133', '134', '135', '151', '152', '153', '154', '155', '156', '157', '171', '172', '173', '174', '175', '176', '201', '202', '203', '204', '205', '206', '207', '209', '211', '212', '214', '218', '219', '221', '222', '224', '225', '226', '227', '228', '229', '230', '231', '233', '234', '251', '252', '253', '254', '255', '256', '257', '258', '271', '272', '273', '274', '999']
⏳ Đang phân tích dữ liệu gán thực tế của Q4...
📊 [Q4] Bộ dải mã thực tế ghi nhận trên các dòng: ['101', '102', '103', '104', '105', '106', '107', '108', '109', '110', '111', '112', '113', '114', '115', '116', '117', '118', '119', '198', '199']

🚀 Bướ